# Notebook 00 — Exploratory Data Analysis
**Run this FIRST. Understand your data before building anything.**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')
os.makedirs('outputs', exist_ok=True)
sns.set_theme(style='darkgrid')
print('Imports done')

## 1. Load & Filter Training Data

In [ ]:
# KAGGLE PATH:
# df = pd.read_csv('/kaggle/input/delhivery-dataset/delivery_data.csv')
df = pd.read_csv('data/delivery_data.csv')

print(f'Total rows: {len(df):,}')
print(f'Data split: {df["data"].value_counts().to_dict()}')

# CRITICAL FIX: filter to training rows only
df = df[df['data'] == 'training'].copy().reset_index(drop=True)
print(f'Training rows: {len(df):,}')
df.head(3)

## 2. Column Overview + Missing Values

In [ ]:
summary = pd.DataFrame({
    'dtype':      df.dtypes,
    'null_count': df.isnull().sum(),
    'null_%':     (df.isnull().mean() * 100).round(2),
    'unique':     df.nunique(),
})
print(summary.to_string())

## 3. Parse Datetimes & Create Core Features

In [ ]:
for col in ['trip_creation_time', 'od_start_time', 'od_end_time', 'cutoff_timestamp']:
    df[col] = pd.to_datetime(df[col], errors='coerce')

# Remove invalid segment times
df = df[(df['segment_actual_time'] > 0) & (df['segment_osrm_time'] > 0)]

# Core delay features
df['segment_delay_ratio'] = (df['segment_actual_time'] / df['segment_osrm_time']).clip(0.1, 10)
df['is_delayed']          = df['segment_delay_ratio'] > 1.2

# Time features
df['hour']        = df['trip_creation_time'].dt.hour
df['day_of_week'] = df['trip_creation_time'].dt.dayofweek
df['month']       = df['trip_creation_time'].dt.month

# Route type encoding
df['route_type_enc'] = (df['route_type'] == 'FTL').astype(int)

print(f'Clean rows: {len(df):,}')
print(f'Route types found: {df["route_type"].unique().tolist()}')
print(f'Delayed trips: {df["is_delayed"].sum():,} ({df["is_delayed"].mean()*100:.1f}%)')
print()
print('Delay ratio stats:')
print(df['segment_delay_ratio'].describe().round(3))

## 4. Delay Ratio Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['segment_delay_ratio'], bins=60, color='#E24B4A', alpha=0.8, edgecolor='white')
axes[0].axvline(1.0, color='green',  linestyle='--', lw=2, label='OSRM (1.0x)')
axes[0].axvline(1.2, color='orange', linestyle='--', lw=2, label='>20% late threshold')
axes[0].set_xlabel('Delay Ratio (actual / OSRM)')
axes[0].set_ylabel('Trip Count')
axes[0].set_title('Segment Delay Ratio Distribution')
axes[0].legend()

for rt, grp in df.groupby('route_type'):
    axes[1].hist(grp['segment_delay_ratio'], bins=40, alpha=0.6, label=rt, edgecolor='white')
axes[1].axvline(1.2, color='red', linestyle='--', lw=2)
axes[1].set_xlabel('Delay Ratio')
axes[1].set_title('Delay Ratio by Route Type')
axes[1].legend()

plt.tight_layout()
plt.savefig('outputs/eda_delay_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Cutoff Analysis (New Column)

In [ ]:
# is_cutoff = whether trip hit a cutoff deadline
print('Cutoff breakdown:')
print(df['is_cutoff'].value_counts())
print()

cutoff_delay = df.groupby('is_cutoff')['segment_delay_ratio'].agg(['mean','median','count'])
print('Delay ratio by cutoff status:')
print(cutoff_delay.round(3))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df['is_cutoff'].value_counts().plot(kind='bar', ax=axes[0],
    color=['#1D9E75','#E24B4A'], edgecolor='white', rot=0)
axes[0].set_title('Cutoff vs Non-Cutoff Trips')
axes[0].set_ylabel('Count')

for val, grp in df.groupby('is_cutoff'):
    axes[1].hist(grp['segment_delay_ratio'], bins=40,
                 alpha=0.6, label=f'cutoff={val}', edgecolor='white')
axes[1].axvline(1.2, color='red', linestyle='--')
axes[1].set_title('Delay Ratio: Cutoff vs Non-Cutoff')
axes[1].legend()

plt.tight_layout()
plt.savefig('outputs/eda_cutoff.png', dpi=120, bbox_inches='tight')
plt.show()

## 6. Delay by Hour of Day

In [ ]:
hourly = df.groupby('hour').agg(
    avg_delay   = ('segment_delay_ratio', 'mean'),
    breach_rate = ('is_delayed',          'mean'),
    trip_count  = ('segment_delay_ratio', 'count')
).reset_index()

fig, ax1 = plt.subplots(figsize=(14, 5))
ax2 = ax1.twinx()
ax1.bar(hourly['hour'], hourly['trip_count'], color='#4a90d9', alpha=0.4, label='Trip Count')
ax2.plot(hourly['hour'], hourly['breach_rate']*100, color='#E24B4A',
         lw=2.5, marker='o', label='Breach Rate %')
ax1.set_xlabel('Hour of Day')
ax1.set_ylabel('Trip Count', color='#4a90d9')
ax2.set_ylabel('SLA Breach Rate (%)', color='#E24B4A')
ax1.set_title('Trip Volume and Breach Rate by Hour')
ax1.set_xticks(range(0,24))
lines1,labs1 = ax1.get_legend_handles_labels()
lines2,labs2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, labs1+labs2)
plt.tight_layout()
plt.savefig('outputs/eda_hourly.png', dpi=120, bbox_inches='tight')
plt.show()

## 7. Route Type Comparison

In [ ]:
rt_stats = df.groupby('route_type').agg(
    trip_count  = ('segment_delay_ratio', 'count'),
    avg_delay   = ('segment_delay_ratio', 'mean'),
    breach_rate = ('is_delayed',          'mean'),
    avg_dist    = ('segment_osrm_distance','mean'),
    avg_actual  = ('segment_actual_time',  'mean'),
    avg_osrm    = ('segment_osrm_time',    'mean'),
).round(3)

print('Route Type Comparison:')
print(rt_stats.to_string())

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = ['#4a90d9','#E24B4A']

rt_stats['trip_count'].plot(kind='bar', ax=axes[0], color=colors, edgecolor='white', rot=0)
axes[0].set_title('Trip Count by Route Type')

rt_stats['avg_delay'].plot(kind='bar', ax=axes[1], color=colors, edgecolor='white', rot=0)
axes[1].axhline(1.2, color='red', linestyle='--', label='20% threshold')
axes[1].set_title('Avg Delay Ratio by Route Type')
axes[1].legend()

(rt_stats['breach_rate']*100).plot(kind='bar', ax=axes[2], color=colors, edgecolor='white', rot=0)
axes[2].set_title('Breach Rate % by Route Type')

plt.tight_layout()
plt.savefig('outputs/eda_route_type.png', dpi=120, bbox_inches='tight')
plt.show()

## 8. Correlation Heatmap

In [ ]:
num_cols = [
    'segment_actual_time','segment_osrm_time','segment_osrm_distance',
    'segment_factor','actual_time','osrm_time','osrm_distance','factor',
    'start_scan_to_end_scan','cutoff_factor','actual_distance_to_destination',
    'segment_delay_ratio','hour','day_of_week','route_type_enc'
]
num_cols = [c for c in num_cols if c in df.columns]
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(13, 10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, square=True, ax=ax, linewidths=0.5,
            cbar_kws={'shrink':0.8})
ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/eda_correlation.png', dpi=120, bbox_inches='tight')
plt.show()

## 9. Top Source & Destination Hubs

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

df['source_name'].value_counts().head(15).plot(
    kind='barh', ax=axes[0], color='#1D9E75', edgecolor='white')
axes[0].set_title('Top 15 Source Hubs by Volume')
axes[0].invert_yaxis()

df['destination_name'].value_counts().head(15).plot(
    kind='barh', ax=axes[1], color='#4a90d9', edgecolor='white')
axes[1].set_title('Top 15 Destination Hubs by Volume')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('outputs/eda_top_hubs.png', dpi=120, bbox_inches='tight')
plt.show()

## 10. EDA Summary

In [ ]:
print('='*55)
print('  EDA SUMMARY')
print('='*55)
print(f'  Training rows:              {len(df):,}')
print(f'  Delayed trips (>20%):       {df["is_delayed"].sum():,} ({df["is_delayed"].mean()*100:.1f}%)')
print(f'  Avg segment delay ratio:    {df["segment_delay_ratio"].mean():.3f}x')
print(f'  Median segment delay ratio: {df["segment_delay_ratio"].median():.3f}x')
print(f'  Unique source hubs:         {df["source_center"].nunique()}')
print(f'  Unique destination hubs:    {df["destination_center"].nunique()}')
print(f'  Route types:                {", ".join(df["route_type"].unique())}')
print(f'  Cutoff trips:               {df["is_cutoff"].sum():,} ({df["is_cutoff"].mean()*100:.1f}%)')
print('='*55)